In [1]:
from envs import set_notebook_env_ids, change_exp_dir
set_notebook_env_ids(
    cuda_version="12.5"
)

import torchvision.transforms as transforms
import torch
import os
import json
from argparse import Namespace
import pprint
import sys

code_dir = "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/SOTA_encoders_StyleGAN/FeatureStyleEncoder/"

Paths = {
        "code_dir": "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/SOTA_encoders_StyleGAN/StyleFeatureEditor-CS",
        "encoder_path": "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/SOTA_encoders_StyleGAN/FeatureStyleEncoder/",
        "images_x_path" : "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/special_images/background", 
        "images_y_path" : "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/special_images/glasses", 
        "device": "cuda",
}

# # # Always stay in evaluation directory
# change_exp_dir(Paths["code_dir"])  # optional, or ensure you launch from there

# sys.path.append(code_dir)
# sys.path.append(".")
# sys.path.append("..")
# change_exp_dir(Paths["code_dir"])
# device = "cuda"

# sys.path.append(code_dir)

CUDA environment variables set for CUDA 12.5
Set TORCH_CUDA_ARCH_LIST to 8.0


In [2]:
import torch
from argparse import Namespace
import pprint
import importlib.util
import os
import yaml

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Define paths to FeatureStyleEncoder assets ===
feature_styleencoder_root = "/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/SOTA_encoders_StyleGAN/FeatureStyleEncoder"
mlp3d_path = os.path.join(feature_styleencoder_root, "models/cs_mlp/mlp3D.py")
trainer_path = os.path.join(feature_styleencoder_root, "trainer.py")
config_path = os.path.join(feature_styleencoder_root, "configs/001.yaml")
checkpoint_path = os.path.join(feature_styleencoder_root, "results/12layers_ls256_lr0.001_wf/checkpoints/iteration_80000.pt")

# === Dynamic import: mlp3D.py ===
spec_mlp = importlib.util.spec_from_file_location("fs_mlp3D", mlp3d_path)
fs_mlp3D = importlib.util.module_from_spec(spec_mlp)
spec_mlp.loader.exec_module(fs_mlp3D)
MappingNetwork_cs_3Dmlp = fs_mlp3D.MappingNetwork_cs_3Dmlp

# === Dynamic import: trainer.py ===
spec_trainer = importlib.util.spec_from_file_location("fs_trainer", trainer_path)
fs_trainer = importlib.util.module_from_spec(spec_trainer)
spec_trainer.loader.exec_module(fs_trainer)
Trainer = fs_trainer.Trainer

# === Load opts from checkpoint ===
print(f'🔄 Loading opts from checkpoint: {checkpoint_path}')
previous_train_ckpt = torch.load(checkpoint_path, map_location='cpu')
opts = Namespace(**previous_train_ckpt['opts'])
pprint.pprint(vars(opts))

# === Load config
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

# === Load Trainer
trainer = Trainer(config, opts)
trainer.initialize(opts.stylegan_model_path, opts.arcface_model_path, opts.parsing_model_path)
trainer.to(device)
trainer.enc.load_state_dict(torch.load(opts.pretrained_model_path))
trainer.enc.eval()

# === Load CS MLP
cs_mlp_net = MappingNetwork_cs_3Dmlp(opts).to(device)
cs_mlp_net.load_state_dict(previous_train_ckpt["state_dict_cs_net"])


ModuleNotFoundError: No module named 'pixel2style2pixel'